# Seminar HCI and BCI in practice
## Session 1 Introduction to data structure and variables

This seminar is based on data from an experiment in which epilepsy patients with subdurally implanted electrode arrays **(ECoG)** performed different hand gestures to control an avatar through a virtual reality.

 The aim of the original project was to detect gestures from the recorded brain signals. For further information, please read following section ```Experimental paradigm and data structure``` 

 For this seminar we took one of these gestures (wiggling with the index finger) and divided it into two antagonistic parts (flexion and extension).

---

## Experimental paradigm and data structure

Most movement related Brain-Machine-Interface (BMI) research in humans has focused on decoding two or three classes of rather elementary real or imagined movements such as sticking out the tongue or squeezing the hand. However, to be able to control more complex devices with more degrees of freedom one would like to be able to discriminate more movement classes. In humans, gestures of the hand and the arm are increasingly used for computer interfacing. However, little is currently known about the feasibility of using different complex gestures to control BMIs. The aim of this study was to investigate whether subdural ECoG-recordings allow for reliable single trial multi-class discrimination between different gestures as well as different phases within gestures.

**Subjects**

Five patients (S1 – S5) with subdurally placed electrode grids for localization of epileptic foci participated in this study. The location of the grid placement was determined only on medical considerations. Center-to-center electrode spacing was either 1 cm (8x8 electrodes) or 0.4 cm (16x16 electrodes).

This course will focus on the data set from one subject only. The data set from this subject originally includes 256 electrodes with an interelectrode spacing of 0.4 cm. In this course however, we will focus on only 40 of these electrodes, in order to reduce the amount of data.

<img src="./docs/figs/fig1_Electrode_Grid.png" width="1000"/>

**Task**

The patients' task was to use gestures of one hand to navigate through a virtual reality (VR) environment to collect tokens. Gestures were assessed with a data glove equipped with one bend sensor for each finger and a three-axis accelerometer. They performed five different gestures to navigate through the VR-environment: Index finger wiggle, waving the hand up and down with extended fingers, turning the hand palm up and down with extended fingers, turning the fist and waving the fist up and down. Based on the glove sensor data we manually assigned time intervals to gestures.

<img src="./docs/figs/fig2_Experimental.png" width="1000"/>

<img src="./docs/figs/fig3_Gesture_Coding.png" width="1000"/>

**This seminar will only focus on the discrimination of flexion and extension (the two antagonistic phases) of the index finger.**

**Data structure**

The experiment originally resulted in two datasets

1. Neurophysiological recordings (#Channles x #Samples) -> rawEcog.pkl

2. Behavioural data (data glove) -> gloveResamp.pkl

The ‘rawEcog.pkl’ data has already been transformed and all the data relevant for the course is saved in the file ecogStruct.pkl.

---

## Define your own working path

In [ ]:
import os
import sys

# Self-defined functions
os.path.join(os.getcwd(), "src")


In [ ]:
# Environment Setting
import scipy.io
import numpy as np
import json
import os
import sys
import pickle
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Self-defined functions
sys.path.append(os.path.join(os.getcwd(), "src"))
from plot_raw_gesture import plot_raw_gesture
from plot_glove_data import plot_glove_data
from plot_own_labels import plot_own_labels

# Define current working directory as your main_path (where you downloaded the seminar scripts and data)
main_path = os.getcwd()
data_path = os.path.join(main_path, 'data','raw')
print(f'Now you are located: {main_path}')

---

## Test glove data set
To make you more familiar with the data you get from a sensor glove we will first look at a test glove data set, that was recorded in order to gain information about the time course of each sensor while performing a particular gesture.

In [ ]:
%matplotlib inline

In [ ]:
# Load test data (consisting of the variable gloveResamp)
glove_file = os.path.join(data_path, 'gloveResamp_Dav_2011_9_7_14_28_21.pkl')
with open(glove_file, 'rb') as f:
    gloveResamp = pickle.load(f)

print(gloveResamp.keys())

# now plot the test glove data
plot_raw_gesture(gloveResamp)
# Upper subplot: data from the sensors of the glove (For us the blue trace is the sensor of interest, as it indicates index finger wiggling)
# Lower subplot: Accelerometer values (Cartesian coordinates)

---

<h2 style="color: #FF0000; font-weight: bold;">TASK 1 (1 Point):</h2>

Try to find the interval in which the subject did the finger wiggling gesture. There are four other gestures (See `Experimental paradigm and data structure`). Can you discriminate them? Write down the gestures you can identify and their approximate time.

<h3 style="color: #FF0000; font-weight: ;">Your Answers or Code demostration:</h3>

**Approach.** Reading the intervals off the figure by eye is imprecise, so instead I used the field `stimOnsets` contained in the test data set: it holds 10 values in microseconds, i.e. the start and end of the five gesture blocks. I then characterised each block with two features that together separate all five gestures of the paradigm:

1. **Finger bend deflection relative to rest** — tells whether the fingers are *extended* (deflection ≈ 0) or the hand is *fisted* (deflection ≫ 0).
2. **Standard deviation of pitch vs. roll** — tells whether the wrist is *waving up and down* (pitch dominant) or *turning* (roll dominant).

The code cell below computes both.

In [ ]:
# --- TASK 1: evidence-based identification of the five gestures ---
# The test file contains the field `stimOnsets`, marking the start and end of each
# gesture block. Its unit is the same as that of `timebase`, i.e. MICROSECONDS:
# timebase runs 0 ... 93,385,000 in steps of 5000 us (= 5 ms = 200 Hz), and every
# stimOnset value is an exact multiple of 5000 lying inside that range.

fingers = np.array(gloveResamp['fingers'])            # (5, n) bend sensors; row 3 = index
pitch   = np.array(gloveResamp['pitch'])
roll    = np.array(gloveResamp['roll'])
t       = np.array(gloveResamp['timebase']) / 1e6     # true time axis [s]
n       = fingers.shape[1]

blocks  = (np.array(gloveResamp['stimOnsets']) / 1e6).reshape(-1, 2)   # -> (5, 2), in s

# Rest baseline = everything outside the gesture blocks (with a 0.5 s guard band)
rest = np.ones(n, dtype=bool)
for a, b in blocks:
    rest &= ~((t >= a - 0.5) & (t <= b + 0.5))
baseline = fingers[:, rest].mean(axis=1)

print(f"{'blk':>3} {'start[s]':>9} {'end[s]':>8} {'dur[s]':>8} |"
      f" {'index bend':>11} {'other fingers':>14} | {'pitch SD':>9} {'roll SD':>8}")
print('-' * 82)
for i, (a, b) in enumerate(blocks, 1):
    m     = (t >= a) & (t <= b)
    dev   = fingers[:, m].mean(axis=1) - baseline          # deflection from rest posture
    other = np.mean([dev[j] for j in range(5) if j != 3])  # mean of the four non-index sensors
    print(f"{i:>3} {a:9.2f} {b:8.2f} {b - a:8.2f} |"
          f" {dev[3]:+11.1f} {other:+14.1f} | {pitch[m].std():9.1f} {roll[m].std():8.1f}")

print("\nbend deflection >> 0  ->  hand is closed (fist);  ~0  ->  fingers extended")
print("pitch SD > roll SD    ->  waving up/down;          roll SD > pitch SD -> turning")

# Note: plot_raw_gesture builds its x-axis as np.linspace(1, n/200, n), which starts
# at 1 s instead of 0 s. The figure is therefore shifted relative to the true timebase,
# so intervals read off that figure appear up to 1 s too late.
t_plot = np.linspace(1, n / 200.0, n)
print(f"\nplot_raw_gesture axis offset vs. true time: "
      f"{(t_plot - t).max():.2f} s at the start, {(t_plot - t)[-1]:.2f} s at the end")

**Answer to the question asked:** the subject performed the **index-finger wiggling gesture between 58.17 s and 68.60 s** (block 4).

It is identifiable because it is the only block in which the index bend sensor deflects strongly (+28.0) while the other four sensors stay at their rest value (−0.4): the finger moves *independently*. It is also the only block in which the wrist is essentially still — pitch SD is 3.6 here versus 15–63 in every other block, roughly a 15-fold difference. So the movement is generated purely by the finger, not by the hand.

**Yes, all four remaining gestures can be discriminated.** They are separated by a 2 × 2 scheme — *fingers extended vs. fisted* crossed with *pitch-dominant (waving) vs. roll-dominant (turning)*. This matches the naming used in `Experimental paradigm and data structure`, where the gestures are literally called *Pitch hand / Pitch fist / Roll hand / Roll fist / Finger*:

| Block | Interval [s] | Fingers | Dominant axis | Gesture (name in the paradigm) |
| :--- | :--- | :--- | :--- | :--- |
| 1 | 13.42 – 23.00 | extended (−0.7) | pitch (54.3 vs. 12.7) | Waving the hand up and down with extended fingers — **Pitch hand** |
| 2 | 28.75 – 38.25 | extended (−1.0) | roll (76.1 vs. 14.6) | Turning the hand palm up and down with extended fingers — **Roll hand** |
| 3 | 43.50 – 53.75 | fisted (+29.2) | pitch (63.1 vs. 16.5) | Waving the fist up and down — **Pitch fist** |
| 4 | 58.17 – 68.60 | index only (+28.0) | neither (3.6 / 2.8) | **Finger** — index finger wiggling |
| 5 | 74.15 – 83.30 | fisted (+30.7) | roll (69.8 vs. 19.4) | Turning the fist up and down — **Roll fist** |

**Remarks.**

- Blocks 2 and 5 look almost identical in pitch/roll — both are roll-dominant rotations. The *only* feature that tells them apart is the finger bend: the hand is open in block 2 and closed in block 5. The same holds for blocks 1 and 3. Without the bend sensors these four gestures would collapse into two classes, which is why the glove needs both sensor types.
- Each gesture lasts about 9–10 s, and the blocks are separated by roughly 5 s of rest.
- The intervals above are given in **true recording time**, taken from `timebase`. Note that `plot_raw_gesture` builds its x-axis as `np.linspace(1, n/200, n)`, which starts at 1 s instead of 0 s; the figure is therefore shifted by up to 1 s, and intervals read off it by eye appear correspondingly too late.
- `pitch` and `roll` are orientation angles derived from the three-axis accelerometer — the glove carries one bend sensor per finger and a three-axis accelerometer, and no gyroscope, even though the subplot in `plot_raw_gesture` is titled "Gyroscope and Fingerbend Sensors".

## Analyzing the glove data of our subject

Now continue with the data from our subject:

In [ ]:
# First load the following data files to your workspace: 

with open(os.path.join(data_path, 'gloveResamp.pkl'), 'rb') as f:
    gloveResamp = pickle.load(f)  # Motion data from the sensor glove

with open(os.path.join(data_path, 'ecogAnalog.pkl'), 'rb') as f:
    ecogAnalog = pickle.load(f)   # contains data from an analog channel to synchronize brain and glove data 
# (it is important to synchronize when using the glove data to find the gesture onsets, that are later used to analyze the brain data)

with open(os.path.join(data_path, 'epoch.pkl'), 'rb') as f:
    epoch = pickle.load(f)        # provides onsets and labels of gestures (hand labeled) 

The gesture onsets and labels contained in the variable epoch were found by hand labeling. 

In *Task 2* you will try this hand labeling yourself.
**BEFORE** you start with the task, plot the hand labeled data saved in epoch, so you know how it will look like when finished (We will plot the glove data time series with onsets of flexion and extension of the index finger.)

In [ ]:
plot_glove_data(gloveResamp,ecogAnalog,epoch)

**Notes:**  As you can see it is hard to recognize anything in this figure. It is best to zoom in using the **zoom tool (magnifying glass)**. Click on the zoom-in symbol (magnifying glass) and choose a time window in your figure. If you want to scroll to other time windows, click on the **coordinate system symbol (scrolling)**

The vertical lines represent the onsets of the epochs (**normal blue lines = flexion epochs; dashed blue lines = extension epochs**) The epochs will be **0.25 seconds** long. If the time of flexion or extension is longer than 0.5 seconds than two epochs will be fitted within one gesture. For better understanding of how the onsets are calculated, have a look at the `getEpochs` function.

Now close this figure again and try finding your own epochs.

<h2 style="color: #FF0000; font-weight: bold;">TASK 2:</h2>

Remember the following steps are just to give you an idea of the procedure used to cut and label the data. Afterwards we will continue using the onsets and labels already saved in epoch.

In order to cut and label the data the function ``` getEpochs.py ``` is used. This function first asks you for graphical input, that you will provide by mouse clicks on the figure that will appear. To make own epochs **select the beginning, the reversal point and the end of one gesture cycle (flexion + extension) with the mouse** (crosshairs will appear to make this easier). **Mark always three points otherwise data are rejected**. From this input the function then calculates flexion and extension onsets. After finishing one gesture, you will be asked if you want to continue with a next gesture, type either ```'y'(es)``` to continue or ```'n'(o)``` to end process. **IMPORTANT** is to optimize your figure size (zoom-in like described above) before starting with the first gesture. 

*Try it out for at least five gestures.*

In [ ]:
# Before marking your own epochs, save the necessary data for later access in the terminal.
# (Run in Terminal, as the Notebook kernel runs extremely slow for this task)
with open(os.path.join(data_path, "data_for_own_epoch.pkl"), "wb") as f:
    pickle.dump({"gloveResamp": gloveResamp, "ecogAnalog": ecogAnalog}, f)

print("✅ Data successfully saved to `data_for_own_epoch.pkl` in main_path, ready for terminal access.")

<div 
    style="border: 1px dashed black; border-radius: 10px; padding: 10px;">
    
## HOWTO: Run `get_epochs.py`

### Arguments:
| Argument | Short | Default                             | Description |
| :--- | :--- |:------------------------------------| :--- |
| `--input` | `-i` | `./data/raw/data_for_own_epoch.pkl` | Path to the input data file. |
| `--output` | `-o` | `./results/epochs.pkl`              | Path where the labels will be saved. |
| `--epoch-length`| `-l` | `0.25`                              | Target length of each epoch in seconds. |
| `--keep-markers`| `-k` | `False`                             | If set, markers from previous clicks remain on screen. |

### Usage:
1. Open your terminal
2. Navigate to the directory of the folder 'BCI_2026'
3. Switch to the Anaconda environment where you have installed the necessary dependencies (e.g., `conda activate BCI_Seminar_2026`)
4. Run the following command:
```bash
python get_epochs.py --input ./data/raw//data_for_own_epoch.pkl --output .results/epochs.pkl --epoch-length 0.25 --keep-markers False
```

or simply:
```bash
python get_epochs.py
```
</div>

In [ ]:
#Code lines for running get_epochs.py(Copy the output and run in a new anaconda terminal)
print('cd', main_path)
print('python get_epochs.py')

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

**Approach.** The labelling has to run outside the notebook, because the interactive figure is unusably slow under the notebook kernel. The cell above writes `gloveResamp` and `ecogAnalog` to `data/raw/data_for_own_epoch.pkl`, and the labelling was then done from a terminal with the Anaconda environment active:

```bash
conda activate BCI_Seminar
python src/get_epochs.py -i ./data/raw/data_for_own_epoch.pkl -o ./results/epochs.pkl -l 0.25
```

This differs from the command printed in the HOWTO box above, which does not run as given: `get_epochs.py` lives in `src/`, not in the repository root; `--keep-markers False` is rejected by `argparse`, because the flag is declared `action="store_true"` and therefore takes no value; and the output path `.results/epochs.pkl` is missing a separator.

In the figure that opens, the trace of interest is the blue one (`fingers[3]`, the index-finger bend sensor), and its x-axis is in **samples**, not seconds. After zooming in, three points were marked per gesture cycle - the start of the flexion, the reversal point, and the end of the extension - for five consecutive cycles. Two practical observations are worth recording, because both look like a broken program rather than intended behaviour:

- Clicks are **silently ignored while the zoom or pan tool is still active** in the toolbar (`_safe_ginput`, line 41), so the tool has to be deselected before marking.
- A half-cycle shorter than the epoch length is **silently discarded**, because `get_epochs` keeps only intervals of at least `srate * len_interval` samples and the corresponding warning is commented out at line 127.

The cell below reloads the resulting file and plots it inline, so that the outcome of the manual labelling is visible in the saved notebook - `plot_own_labels` forces the TkAgg backend and opens a separate window, which leaves no output behind.

In [ ]:
# --- TASK 2: reload and inspect the epochs I labelled by hand in the terminal ---
%matplotlib inline

with open(os.path.join(main_path, 'results', 'epochs.pkl'), 'rb') as f:
    ownEpochs = pickle.load(f)

onsets = np.array(ownEpochs['OnsetIdx'])
labels = np.array(ownEpochs['label'])
srate  = gloveResamp['srate']
shift  = -41886                      # same alignment constant used by get_epochs.py

print(f"epochs labelled by hand : {len(labels)}")
print(f"  flexion   (label 20)  : {(labels == 20).sum()}")
print(f"  extension (label 21)  : {(labels == 21).sum()}")
print(f"onset range             : {onsets.min()} - {onsets.max()} samples"
      f"  ({(onsets.max() - onsets.min()) / srate:.1f} s of recording)")

t = np.arange(len(gloveResamp['gesture'])) + shift

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(np.round(np.array(ecogAnalog) * 10), label='Analog')
ax.plot(t, gloveResamp['fingers'][3], 'b', label='Finger (index bend)')
ax.plot(t, gloveResamp['pitch'], 'r', alpha=0.4, label='Pitch')

for idx, lab in zip(onsets, labels):
    ax.axvline(idx, color='k', linestyle='-' if lab == 20 else ':', linewidth=1)

ax.set_xlim(onsets.min() - 1500, onsets.max() + 1500)
ax.set_title('TASK 2: my own hand-labelled epochs '
             '(solid = flexion onset, dotted = extension onset)', fontweight='bold')
ax.set_xlabel('Samples', fontweight='bold')
ax.set_ylabel('Amplitude', fontweight='bold')
ax.legend(loc='upper right')
plt.show()

**Result.** Five gesture cycles were marked, spanning samples 84,785 to 96,668, i.e. about 11.7 s of the recording. From them `get_epochs` derived **9 epochs of 0.25 s: 5 flexion (label 20) and 4 extension (label 21)**.

The count is 5 + 4 rather than 5 + 5 because one extension half-cycle - the interval between my second and third click of that gesture - was shorter than the epoch length of 0.25 s (about 254 samples at 1017.25 Hz) and was removed by the length filter. This is the documented behaviour of the function rather than a failed selection, but it illustrates a limitation of the method: how many epochs survive depends both on how quickly the subject moved and on how precisely the reversal point was clicked.

Comparing the figure above with the provided labelling plotted earlier, the onsets sit on the same features of the blue index-bend trace - solid lines where the finger starts to bend, dotted lines where it starts to straighten again. The manual procedure therefore reproduces the given labelling, but it is slow and its precision is bounded by the click accuracy, which is why the analysis continues with the onsets already stored in `epoch`.

<h3 style="color: #FF0000; font-weight: bold;">TASK 2.1 (1 Point)</h3>

Find the onsets of the 5 gestures you defined. Copy them in here:


<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

The five marked gesture cycles yielded the nine onsets below, printed by the cell above from `ownEpochs['OnsetIdx']` and `ownEpochs['label']`. They are stored as **sample indices on the analog time axis**, which is the axis the clicks were made on. To convert one into recording time, the alignment constant of `get_epochs.py` has to be undone first and the glove sampling rate applied: `t = (OnsetIdx + 41886) / 1017.25 Hz`.

| # | Label | Type | Onset [samples] | Time [s] |
| :--- | :--- | :--- | ---: | ---: |
| 1 | 20 | flexion | 84 785 | 124.52 |
| 2 | 21 | extension | 85 082 | 124.81 |
| 3 | 20 | flexion | 85 544 | 125.27 |
| 4 | 21 | extension | 85 864 | 125.58 |
| 5 | 20 | flexion | 89 540 | 129.20 |
| 6 | 20 | flexion | 95 541 | 135.10 |
| 7 | 21 | extension | 95 885 | 135.43 |
| 8 | 20 | flexion | 96 371 | 135.91 |
| 9 | 21 | extension | 96 668 | 136.20 |

Each onset marks the **start of a 0.25 s epoch**, not the whole movement. Label 20 denotes a flexion epoch and label 21 an extension epoch.

The gestures are not evenly spaced: cycles 1-2 and 4-5 follow each other within roughly a second, while gesture 3 sits alone at 129.20 s. That is a property of the recording, in which the subject wiggled the finger in short bursts, and not an artefact of the selection. Gesture 3 also contributed only a flexion epoch, because its extension half-cycle fell below the 0.25 s minimum length.

In [ ]:
with open(os.path.join(data_path, "data_for_own_epoch.pkl"), "wb") as f:
    pickle.dump({"gloveResamp": gloveResamp, "ecogAnalog": ecogAnalog}, f)

print("✅ Data saved to data_for_own_epoch.pkl")

In [ ]:
import os
import pickle

main_path = os.getcwd()

In [ ]:
with open(os.path.join(main_path, "results", "epochs.pkl"), "rb") as f:
    myEpoch = pickle.load(f)

print(type(myEpoch))
if isinstance(myEpoch, dict):
    print(myEpoch.keys())     # pehle keys dekho
print(myEpoch)

In [ ]:
# After label your own epochs, now load the labeled epochs
with open(r".\results\epochs.pkl", "rb") as f:
    ownEpochs = pickle.load(f)

# check data
print("loaded own labeled epoch data:", ownEpochs)

# Let's have a look of your own labeled epochs
plot_own_labels(gloveResamp,ecogAnalog,ownEpochs)

---

<h2 style="color: #FF0000; font-weight: bold;">TASK 3 (Discussion, 1 Point):</h2>

Check if only the index finger moved in the periods you labeled. Think about the appropriateness of the label if more than one finger moved. 


<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

**Approach.** Whether the label is appropriate cannot be judged from the figure, because the four remaining bend sensors are drawn on the same axis as a signal ten times larger. The cell below therefore measures it: for every one of the nine labelled epochs it computes the peak-to-peak deflection of all five bend sensors over the 0.25 s the epoch covers, and expresses the result as the ratio between the index sensor and the strongest of the other four.

The second half of the cell adds the context the per-epoch view leaves out. It compares the variability of each sensor **inside** the labelled epochs with its variability over the **whole stretch** those epochs lie in - the gaps between the epochs included - and correlates each sensor with the index sensor over that stretch.

In [ ]:
# --- TASK 3: did only the index finger move inside the labelled epochs? ---
F = np.array(gloveResamp['fingers'])          # (5, n) bend sensors; row 3 = index
L = int(round(srate * 0.25))                  # epoch length in samples
order = np.argsort(onsets)

print("peak-to-peak deflection of every bend sensor inside each labelled epoch")
print(f"{'ep':>3} {'type':>10} {'onset':>7} |"
      + "".join(f"{'f' + str(i) + ('*' if i == 3 else ''):>8}" for i in range(5))
      + f" |{'ratio':>7}")
print('-' * 74)
for k, j in enumerate(order, 1):
    s = onsets[j] - shift
    rng = np.ptp(F[:, s:s + L], axis=1)
    other = max(rng[i] for i in range(5) if i != 3)
    print(f"{k:>3} {'flexion' if labels[j] == 20 else 'extension':>10} {onsets[j]:>7} |"
          + "".join(f"{v:8.1f}" for v in rng) + f" |{rng[3] / other:6.1f}x")
print("\n* f3 is the index-finger sensor; 'ratio' is index / strongest other finger")

# Same sensors, but inside the epochs versus over the whole stretch they lie in
inside = np.zeros(F.shape[1], dtype=bool)
for s in onsets - shift:
    inside[s:s + L] = True
lo, hi = (onsets - shift).min(), (onsets - shift).max() + L

print(f"\npooled over all {inside.sum()} labelled samples vs. the whole stretch they lie in:")
print(f"{'sensor':>8} {'SD inside':>10} {'SD stretch':>11} {'r with f3':>10}")
for i in range(5):
    r = np.corrcoef(F[i, lo:hi], F[3, lo:hi])[0, 1]
    print(f"{'f' + str(i):>8} {F[i][inside].std():10.2f} {F[i, lo:hi].std():11.2f} {r:+10.2f}")

**Answer: inside the labelled periods only the index finger moved, so the label is appropriate.**

In every one of the nine epochs the index sensor deflects by 20.5 to 33.9 units, while the strongest of the other four sensors stays between 0.9 and 4.8. The ratio never drops below **7:1** and reaches 22:1; the worst case is epoch 1, where the middle-finger sensor (f2) picks up 4.8 units against 33.9 for the index. Sensors f1 and f4 are essentially flat throughout, frequently below 0.5 units, which is the noise level of the glove. Nothing in the labelled windows is consistent with a second finger contributing to the movement.

**The picture outside the epochs is different, and it matters for the answer.** Pooled over the 2286 labelled samples the other sensors have a standard deviation of only 0.2 to 1.3, but over the whole stretch those epochs lie in - the pauses between the gestures included - it rises to 2.6 to 4.9, and every sensor correlates positively with the index sensor at r = +0.52 to +0.70. So the fingers are not mechanically independent: they share tendons and the hand posture drifts along with the index finger between the gestures. What makes the label sound is not that the other fingers never move, but that the **0.25 s windows were placed on the part of the cycle where the index movement dominates**.

**On the appropriateness of the label if more than one finger had moved.** The label claims that a specific, isolated movement occurred, and the ECoG epochs cut around these onsets are later averaged as if every one of them contained the same event. If a second finger moved as well, that claim would break in two ways:

- **The label becomes ambiguous.** Cortical representations of neighbouring fingers on the motor strip lie only a few millimetres apart, well within the reach of a 16 x 16 grid at 0.4 cm spacing. An epoch containing index *and* middle finger movement would carry activity from both, but would be averaged into the class "index".
- **The direction of the error is not random.** A co-moving finger tends to move in the *same* phase as the index finger, as the correlations above show, so its contribution does not average out over epochs - it adds a systematic component to the class mean, which is far more damaging than noise.

A more honest treatment would be to keep the epoch but relabel it, for instance as a combined or a rejected trial, rather than to record a single-finger label for a multi-finger movement. Applying a rejection criterion of the kind used here - requiring the index deflection to exceed the strongest other sensor by some fixed factor - would make that decision explicit and reproducible instead of leaving it to visual judgement.

---

## Anatomy
Now have a short look at the anatomy of our subject. Raw ECoG data were recorded from a 16x16 electrode grid with a center-to-center spacing of 0.4 cm. To reduce the amount of data, only 40 of the 256 electrodes available, will be used in this course.

Plot anatomical image of brain and electrodes

<h2 style="color: #FF0000; font-weight: bold;">TASK 4 （1 Point):</h2>

Figure out how to read in and plot the anatomical data in Jupyter Notebook

`Anatomy = './docs/figs/GP33_anatomy_40_electrodes.png' `

In [ ]:
# Read and show anatomical picture
Anatomy = './docs/figs/GP33_anatomy_40_electrodes.png'
img_path = os.path.join(main_path, Anatomy)

img = mpimg.imread(img_path)



<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>

In [ ]:
# Read and show anatomical picture
Anatomy = './docs/figs/GP33_anatomy_40_electrodes.png'
img_path = os.path.join(main_path, Anatomy)

img = mpimg.imread(img_path)


# plot show
plt.figure(figsize=(20, 20))
plt.imshow(img)
plt.axis("off")  # axis off
plt.show()
# parameter for range

---

## ECoG data
The data is saved in a `dict` called `ecog`. Most of the functions we use in this seminar come from our ecog toolbox and are based on this kind of data structure.

In [ ]:
with open(os.path.join(data_path, 'ecogStruct.pkl'), 'rb') as f:
    ecog = pickle.load(f) 

# now have a look at the current state of the ecog structure
ecog.keys()

In [ ]:
# Go through the keys of the dictionary 'ecog' and check the class and dimensions of data
for k in ecog.keys():
    print(f'{k}, class: {type(ecog[k]).__name__}, shape: {np.array(ecog[k]).shape}')
# Note: if NumPy returns shape=(), that means 0-dim data, which also means it is a scalar

<h2 style="color: #FF0000; font-weight: bold;">TASK 5 （1 Point):</h2>

Try to understand what all the fields in the ecog structure contain. 

- What are the meanings of each key-value pair in the `dict` `ecog`?
- How is the shape of data under each key?
- How is the data saved in the `dict` `ecog` (data-type/ class)?


<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

In [ ]:
# --- TASK 5: check what is inside the ecog dict ---
ecog_data = np.array(ecog['data'])
tb = np.array(ecog['timebase'])

for k in ecog.keys():
    v = np.array(ecog[k])
    print(f"{k:17} class={type(ecog[k]).__name__:6} shape={str(v.shape):14} dtype={v.dtype}")

print()
print("sampDur * srate     =", ecog['sampDur'] * ecog['srate'], "-> so sampDur is in ms")
print("nSamp               =", ecog['nSamp'],
      "| data.shape[1] =", ecog_data.shape[1],
      "| len(timebase) =", len(tb))
print("timebase first/last =", round(tb[0], 1), "/", round(tb[-1], 1), "ms")
print("recording length    =", round(ecog['nSamp'] / ecog['srate'], 1), "s")
print("selectedChannels    =", np.array(ecog['selectedChannels'])[:5], "...",
      np.array(ecog['selectedChannels'])[-3:])
print("mean of ch 1-3      =", np.round(ecog_data.mean(axis=1)[:3], 1), "-> not zero yet")

**My answer.** `ecog` is a normal Python `dict` with 6 keys. The big fields are saved as plain Python **lists**, not as numpy arrays, so I have to convert them with `np.array()` before I can do maths on them. The three small fields are single numbers, which is why numpy shows their shape as `()`.

| key | class | shape | what it contains |
| :--- | :--- | :--- | :--- |
| `data` | list | (40, 522868) | the ECoG signal: 40 channels, 522868 samples each |
| `timebase` | list | (522868,) | the time of every sample, in ms |
| `sampDur` | float | () = scalar | how long one sample lasts: 0.98304 ms |
| `nSamp` | int | () = scalar | number of samples per channel: 522868 |
| `selectedChannels` | list | (40,) | the electrodes that were kept: 1 to 40 |
| `srate` | float | () = scalar | sampling rate: 1017.25 Hz |

A few things I noticed when I checked the numbers in the cell above:

- `sampDur * srate` gives exactly 1000, so `sampDur` is in **milliseconds** and `timebase` is in ms too.
- `nSamp`, `data.shape[1]` and `len(timebase)` are all 522868, so the three fields fit together. The recording is about 514 s, so around 8.5 minutes.
- `timebase` is **negative**, it goes from -514000 ms up to -1 ms. So the time is counted backwards and 0 is the **end** of the recording, not the beginning.
- `srate` is exactly the same as `gloveResamp['srate']`. The glove data was resampled to the ECoG rate, and that is why the glove onsets can be used directly on the ECoG data.
- `selectedChannels` is just 1, 2, 3 ... 40. These are the 40 electrodes kept out of the 256 of the grid, and they are already renumbered.
- The channel means are not 0 (channel 1 is +271, channel 3 is -597). So the baseline correction has not been done yet, that is what Task 7 does.

---

## Preprocessing of ECoG Data

The first preprocessing step is baseline correction. Here, the mean across the samples within one channel is subtracted from each sample in the respective channel, setting the average of each channel to zero.

<h2 style="color: #FF0000; font-weight: bold;">TASK 6 (Discussion, 1 Point)</h2>

What is the goal of the baseline correction here? Can you think of other ways of achieving this goal?

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

In [ ]:
# --- TASK 6: how big is the offset compared to the real signal? ---
ecog_data = np.array(ecog['data'])

means = ecog_data.mean(axis=1)          # one offset per channel
sds   = ecog_data.std(axis=1)           # how much the signal itself varies

print("channel offsets (means):  min %.1f   max %.1f   spread %.1f"
      % (means.min(), means.max(), means.max() - means.min()))
print("signal variation (SD)  :  median over channels %.1f" % np.median(sds))
print("-> the offset is about %.0f times bigger than the signal itself"
      % ((means.max() - means.min()) / np.median(sds)))

# does the offset also drift during the recording?
half = ecog_data.shape[1] // 2
drift = ecog_data[:, half:].mean(axis=1) - ecog_data[:, :half].mean(axis=1)
print("\ndrift (mean of 2nd half - mean of 1st half): min %.1f  max %.1f"
      % (drift.min(), drift.max()))

# would the median give a different baseline than the mean?
print("largest difference between channel mean and channel median: %.1f"
      % np.abs(means - np.median(ecog_data, axis=1)).max())

**Goal of the baseline correction.** Every channel has its own constant voltage offset that has nothing to do with the brain. It comes from the electrode and the amplifier, and it is different for each electrode. In our data the channel means go from -634 to +828, so the channels are spread over about 1462 units, while the signal itself only varies with an SD of about 47. The offset is therefore roughly **30 times bigger than the activity we actually want to look at**.

By subtracting the mean of each channel from that channel, every channel gets an average of 0. Two things are gained:

- The channels become **comparable**. Without it, channel 3 (mean -597) and channel 2 (mean +420) would look completely different even if they recorded the same brain activity.
- The offset no longer **leaks into later steps**. If we average channels, compute power or variance, or plot them all together, the constant offset would dominate everything and hide the real signal.

It is important that the offset is removed *per channel* and not one value for all channels, because each electrode has its own offset.

**Other ways of achieving the same thing.**

- **Subtract the median instead of the mean.** The median is not pulled around by artifacts or a few extreme samples. In our data it would hardly change anything, because mean and median differ by at most 3.0, but with a noisy channel it would be safer.
- **High-pass filter the data** (for example at 0.1 Hz). This removes the constant offset as well, and it also removes slow drift, which subtracting the mean cannot do. The disadvantage is that a filter also changes the signal a little bit.
- **Detrending**, so fitting a straight line (or a polynomial) to each channel and subtracting it. This handles the case where the offset slowly changes during the recording. In our data this is not really needed: the mean of the second half differs from the first half only by 1.2 to 5.8, so the offset stays almost constant.
- **Baseline per epoch instead of the whole recording.** Instead of using all 522868 samples, one can take a short window before each movement onset and subtract its mean from that epoch. This is what is normally done for event-related analysis, because it also removes slow changes that happen between the trials.
- **Re-referencing, for example a common average reference**, where at every time point the mean over all channels is subtracted. This removes the offset too, but it does more than that: it also removes noise that all electrodes share. It changes what the signal is measured against, so it is not just a baseline correction.

<h2 style="color: #FF0000; font-weight: bold;">TASK 7 (2 pt)</h2>

Implement two ways of performing baseline correction.
1. Using a for-loop over channels. (If you want to get some challenge, try list comperhension instead of a for-loop)
2. Creating a vector of channel means and subtracting them from all samples simultaneously.

<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>

In [ ]:
# Baseline Correction
ecog_data = np.array(ecog['data'])

# Method 1: go through the channels one by one with a for-loop
ecog_bc_1 = np.zeros_like(ecog_data)
for i in range(ecog_data.shape[0]):
    ecog_bc_1[i] = ecog_data[i] - ecog_data[i].mean()

# Method 2: make a vector of the channel means and subtract it from everything at once
ecog_bc_2 = ecog_data - ecog_data.mean(axis=1, keepdims=True)

# check that both methods give the same thing
print("both methods give the same result:", np.allclose(ecog_bc_1, ecog_bc_2))
print("biggest channel mean before:", round(np.abs(ecog_data.mean(axis=1)).max(), 1))
print("biggest channel mean after :", np.abs(ecog_bc_1.mean(axis=1)).max())

# Extracting a sample channel
before_correction = ecog['data'][0][100000:110000]  # Original data
after_correction = ecog_bc_1[0][100000:110000]
time_points = np.array(ecog['timebase'])[100000:110000]

# Create the figure with two subplots
fig, axs = plt.subplots(2, 1, figsize=(10, 6))

# First subplot: Before baseline correction
axs[0].plot(time_points, before_correction)
axs[0].set_title('Before baseline correction')
axs[0].set_xlabel('Time')
axs[0].set_ylabel('Amplitude')

# Second subplot: After baseline correction
axs[1].plot(time_points, after_correction)
axs[1].set_title('After baseline correction')
axs[1].set_xlabel('Time')
axs[1].set_ylabel('Amplitude')

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

**My answer.** Both methods do exactly the same thing, they only differ in how the subtraction is written.

**Method 1 (for-loop).** I first make an empty array of the same shape with `np.zeros_like`, then I go through the 40 channels one by one, take the mean of that channel and subtract it from that channel. This is easy to read, because it does exactly what the text describes: one channel at a time.

The same thing as a list comprehension is a bit shorter:

```python
ecog_bc_1 = np.array([ch - ch.mean() for ch in ecog_data])
```

**Method 2 (all at once).** `ecog_data.mean(axis=1, keepdims=True)` gives one mean per channel. `axis=1` means the mean is taken over the samples, so we get 40 values and not one single number. `keepdims=True` keeps the shape as (40, 1) instead of (40,), and numpy then broadcasts it, so every channel gets its own mean subtracted in one step. Without `keepdims` the shape would be (40,) and numpy would try to match it to the 522868 samples, which gives an error.

**Check.** `np.allclose(ecog_bc_1, ecog_bc_2)` is `True`, so the two methods really give the same result. The biggest channel mean before the correction is 827.9 and after the correction it is about 1.7e-13, which is zero apart from floating point rounding.

Method 2 is also the faster one (about 0.18 s against 0.29 s for the loop), because numpy does the subtraction in compiled code instead of looping in Python. With only 40 channels that does not matter much, but with all 256 electrodes it would.

In the figure the two traces have the same shape, the lower one is just shifted down so that it is centred on 0. That is what baseline correction does: it does not change the signal, it only removes the constant offset.